In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression

print("All installed")

All installed


# NHANES 2021-2023: Sleep and Blood Pressure
## Step 1: Load and Merge Data

In [9]:
# Load each XPT file
demo = pd.read_sas('../Data/DEMO_L.xpt', format='xport', encoding='utf-8')
bp_exam = pd.read_sas('../Data/BPXO_L.xpt', format='xport', encoding='utf-8')
bp_q = pd.read_sas('../Data/BPQ_L.xpt', format='xport', encoding='utf-8')
sleep = pd.read_sas('../Data/SLQ_L.xpt', format='xport', encoding='utf-8')
body = pd.read_sas('../Data/BMX_L.xpt', format='xport', encoding='utf-8')
activity = pd.read_sas('../Data/PAQ_L.xpt', format='xport', encoding='utf-8')

In [10]:
# Check how many participants are in each file
print("Demographics:", demo.shape)
print("BP Exam:", bp_exam.shape)
print("BP Questionnaire:", bp_q.shape)
print("Sleep:", sleep.shape)
print("Body Measures:", body.shape)
print("Activity:", activity.shape)

Demographics: (11933, 27)
BP Exam: (7801, 12)
BP Questionnaire: (8501, 6)
Sleep: (8501, 7)
Body Measures: (8860, 22)
Activity: (8153, 8)


In [12]:
df = demo.merge(bp_exam, on='SEQN', how='inner') # only want participants who actually have bp measurements
df = df.merge(bp_q, on='SEQN', how='left')
df = df.merge(sleep, on='SEQN', how='left')
df = df.merge(body, on='SEQN', how='left')
df = df.merge(activity, on='SEQN', how='left')

print("Merged shape:", df.shape)

Merged shape: (7801, 77)


In [13]:
df.head()


,SEQN,SDDSRVYR,RIDSTATR,RIAGENDR,RIDAGEYR,RIDAGEMN,RIDRETH1,RIDRETH3,RIDEXMON,RIDEXAGM,...,BMIWAIST,BMXHIP,BMIHIP,PAD790Q,PAD790U,PAD800,PAD810Q,PAD810U,PAD820,PAD680
0,130378.0,12.0,2.0,1.0,43.0,NaN,5.0,6.0,2.0,NaN,...,NaN,102.9,NaN,3.000000e+00,W,45.0,3.000000e+00,W,45.0,360.0
1,130379.0,12.0,2.0,1.0,66.0,NaN,3.0,3.0,2.0,NaN,...,NaN,112.4,NaN,4.000000e+00,W,45.0,3.000000e+00,W,45.0,480.0
2,130380.0,12.0,2.0,2.0,44.0,NaN,2.0,2.0,1.0,NaN,...,NaN,98.0,NaN,1.000000e+00,W,20.0,5.397605e-79,,NaN,240.0
3,130386.0,12.0,2.0,1.0,34.0,NaN,1.0,1.0,1.0,NaN,...,NaN,110.6,NaN,1.000000e+00,W,30.0,1.000000e+00,M,30.0,180.0
4,130387.0,12.0,2.0,2.0,68.0,NaN,3.0,3.0,2.0,NaN,...,NaN,148.9,NaN,5.397605e-79,,NaN,5.397605e-79,,NaN,1200.0


In [20]:
# select only relevant variables
cols = ['SEQN', 'RIDAGEYR', 'RIAGENDR', 'BMXBMI', 'BPXOSY1', 'BPXODI1', 'BPQ020', 'SLD012']


df = df.rename(columns={'RIDAGEYR': 'age','RIAGENDR': 'sex','BMXBMI': 'bmi','BPXOSY1': 'systolic_bp','BPXODI1': 'diastolic_bp','BPQ020': 'hypertension_diagnosis', 'SLD012': 'sleep_hours'})

cols = ['SEQN', 'age', 'sex', 'bmi', 'systolic_bp', 'diastolic_bp', 'hypertension_diagnosis', 'sleep_hours']

df = df[cols]
print(df.shape)
df.head()

(7801, 8)


,SEQN,age,sex,bmi,systolic_bp,diastolic_bp,hypertension_diagnosis,sleep_hours
0,130378.0,43.0,1.0,27.0,135.0,98.0,1.0,9.5
1,130379.0,66.0,1.0,33.5,121.0,84.0,1.0,9.0
2,130380.0,44.0,2.0,29.7,111.0,79.0,2.0,8.0
3,130386.0,34.0,1.0,30.2,110.0,72.0,2.0,7.5
4,130387.0,68.0,2.0,42.6,143.0,76.0,1.0,3.0


In [21]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 7801 entries, 0 to 7800
Data columns (total 8 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   SEQN                    7801 non-null   float64
 1   age                     7801 non-null   float64
 2   sex                     7801 non-null   float64
 3   bmi                     7691 non-null   float64
 4   systolic_bp             7517 non-null   float64
 5   diastolic_bp            7517 non-null   float64
 6   hypertension_diagnosis  6615 non-null   float64
 7   sleep_hours             6550 non-null   float64
dtypes: float64(8)
memory usage: 487.7 KB


In [26]:
df['hypertension_diagnosis'] = df['hypertension_diagnosis'].replace({7: np.nan, 9: np.nan})
df['hypertension_diagnosis'] = df['hypertension_diagnosis'].astype(float)

df.info()
df.describe()


<class 'pandas.DataFrame'>
RangeIndex: 7801 entries, 0 to 7800
Data columns (total 8 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   SEQN                    7801 non-null   float64
 1   age                     7801 non-null   float64
 2   sex                     7801 non-null   float64
 3   bmi                     7691 non-null   float64
 4   systolic_bp             7517 non-null   float64
 5   diastolic_bp            7517 non-null   float64
 6   hypertension_diagnosis  6611 non-null   float64
 7   sleep_hours             6550 non-null   float64
dtypes: float64(8)
memory usage: 487.7 KB


,SEQN,age,sex,bmi,systolic_bp,diastolic_bp,hypertension_diagnosis,sleep_hours
count,7801.000000,7801.000000,7801.000000,7691.000000,7517.000000,7517.000000,6611.000000,6550.000000
mean,136349.487117,44.828740,1.539162,28.307541,119.288546,72.748038,1.647103,7.739542
std,3449.490842,22.722662,0.498496,7.749019,18.561052,11.895572,0.477907,1.598841
min,130378.000000,8.000000,1.000000,11.100000,61.000000,33.000000,1.000000,2.000000
25%,133335.000000,23.000000,1.000000,23.000000,106.000000,64.000000,1.000000,7.000000
50%,136382.000000,47.000000,2.000000,27.200000,117.000000,72.000000,2.000000,8.000000
75%,139325.000000,65.000000,2.000000,32.400000,130.000000,80.000000,2.000000,8.500000
max,142310.000000,80.000000,2.000000,74.800000,232.000000,142.000000,2.000000,14.000000


In [27]:
# Dropping rows with missing values

df_clean = df.dropna()
print("Before:", df.shape)
print("After:", df_clean.shape)
print("Dropped:", df.shape[0] - df_clean.shape[0])

Before: (7801, 8)
After: (6267, 8)
Dropped: 1534


In [28]:
# Filter to adult results only to match the research question
df_clean = df_clean[df_clean['age'] >= 18]
print("Adults only:", df_clean.shape)

Adults only: (5999, 8)


In [29]:
df_clean.to_csv('../Data/nhanes_clean.csv', index=False)

## Summary
- Loaded 6 NHANES 2021-2023 XPT files
- Merged on SEQN, resulting in 7,801 participants
- Selected 8 variables relevant to research question
- Replaced NHANES refusal codes (7, 9) with NaN in hypertension_diagnosis
- Dropped 1,534 rows with missing values
- Filtered to adults 18+, final sample: 5,999 participants